# 02 · EDA: USDA FoodData Central Branded (D2)
Run step3 and step3b first (`data/processed/usda_labeled.parquet`).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "runtime" / "src"))
import pandas as pd
from IPython.display import Image, display
from nutriscan import config, eda
from nutriscan.data import clean_products
pd.set_option("display.max_columns", 40)
print("outputs read from / written to:", config.OUT_ROOT)

In [ ]:
from nutriscan.usda import mapping_table
usda = pd.read_parquet(config.USDA_LABELED)
display(eda.overview(usda, "USDA"))
display(eda.missing_table(usda).head(15))

## Category mapping to OFF food groups (plan Section 4.1)

In [ ]:
display(mapping_table(usda).head(30))
print("share mapped:", round(usda["category"].notna().mean(), 3))
display(Image(eda.plot_counts(usda["category"], "eda_E6_usda_mapped_groups", "USDA products per mapped OFF group", color="#ef6c00")))

## Health tier from USDA nutrients (Nutri-Score 2017, fruit/veg share set to 0)

In [ ]:
tier = usda.loc[usda["health_tier"].isin(eda.TIER_ORDER), "health_tier"]
display(eda.class_balance(tier))
display(Image(eda.plot_class_distribution(tier, "eda_E2_usda_tier", "USDA health tier", order=eda.TIER_ORDER, colors=eda.TIER_COLORS)))

## OFF vs USDA (E10)

In [ ]:
off = eda.add_text_features(pd.read_parquet(config.OFF_CLEAN))
u = eda.add_text_features(usda)
display(Image(eda.plot_compare(off["n_ingredients"], u["n_ingredients"], ("OFF", "USDA"),
                               "eda_E10_off_vs_usda_n_ingredients", "E10  Ingredients per product")))
display(pd.DataFrame({"OFF": off["category"].value_counts(normalize=True),
                      "USDA": u["category"].value_counts(normalize=True)}).round(3))